<a href="https://colab.research.google.com/github/Jeremy26/vslam/blob/main/Visual_SLAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Visual Odometry Workshop**
Welcome to the VO Workshop!

In this workshop, we're going to learn how to use feature tracking to build an odometry estimation algorithm! This is very useful in Visual SLAM systems, often considered Step #1.
<P>
So let's begin! We will do this in 3 steps:
1. Feature Tracking (Detection, Description, Matching)
2. Pose Recovery (E & F, R & T)
3. Visual Odometry Graph

But first, let's do some imports...

## **Waymo Open Dataset & Imports**

In [ ]:
!wget -qq https://optical-flow-data.s3.eu-west-3.amazonaws.com/waymo_images.zip
!unzip -qq waymo_images.zip && rm waymo_images.zip
!mkdir output
!ls

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pickle
from google.colab.patches import cv2_imshow

In [ ]:
#TODO: Load Different Image Pairs
img1 = cv2.imread("downtown/front_images_downtown/1557197711848851.jpg")
img2 = cv2.imread("downtown/front_images_downtown/1557197711948687.jpg")

def to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.imshow(to_rgb(img2))

The sample images will be used to experiment with the feature tracking, but frankly, in Visual SLAM, we need a complete sequence.

In [ ]:
import json

video_path = 'downtown/front_camera_downtown.mp4'
cap = cv2.VideoCapture(video_path)

# Define the path to the camera calibration JSON file
calibration_json_path = 'downtown/camera_calibration_downtown.json'

print("Video file path: ", video_path)
print("Calibration JSON path: ", calibration_json_path)

# Open and read the JSON file
with open(calibration_json_path, 'r') as f:
    calibration_data = json.load(f)

# Extract camera intrinsics matrix and distortion coefficients
camera_matrix = np.array(calibration_data['intrinsics'])
dist_coeffs = np.array(calibration_data['distortion'])

print("Camera Intrinsics Matrix (K):")
print(camera_matrix)
print("\nDistortion Coefficients:")
print(dist_coeffs)

## **vSLAM Process**

In [ ]:
# INITIALIZE ORB DETECTOR/DESCRIPTOR
orb = cv2.ORB_create(nfeatures=2000)

FLANN_INDEX_KDTREE = 0
index_params = dict(algorithm = FLANN_INDEX_KDTREE, trees = 5)
search_params = dict(checks = 50)

flann = cv2.FlannBasedMatcher(index_params, search_params)

print("Images loaded and feature objects initialized.")

In [ ]:
# DETECTOR/DESCRIPTOR IMAGE 1
kp1 = orb.detect(img1, None) #detector
kp1, des1 = orb.compute(img1, kp1) #descriptor
#alternatively: kp1, des1 = orb.detectAndCompute(img1, None)


# DETECTOR
kp2, des2 = orb.detectAndCompute(img2, None)

In [ ]:
img1_kp = cv2.drawKeypoints(img1, kp1, None, color=(0,255,0), flags=0)
img2_kp = cv2.drawKeypoints(img2, kp2, None, color=(0,255,0), flags=0)

fig = plt.figure(figsize=(15, 7))
ax1 = plt.subplot(121)
ax2 = plt.subplot(122)

ax1.imshow(to_rgb(img1_kp))
ax2.imshow(to_rgb(img2_kp))
plt.show()

In [ ]:
# FLANN MATCHING
# The `flann` matcher was already created in the setup cell above, so we just use it.
matches = flann.knnMatch(np.float32(des1), np.float32(des2), k=2)  # NP.FLOAT32 needed for ORB/BRIEF

# Keep only confident matches using Lowe's ratio test
good = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        good.append(m)

print(f"Found {len(good)} good matches after Lowe's ratio test.")

In [ ]:
draw_params = dict(matchColor=(0, 255, 0),   # draw matches in green
                   singlePointColor=None,
                   flags=2)

img_briefmatch = cv2.drawMatches(img1, kp1, img2, kp2, good, None, **draw_params)
cv2_imshow(img_briefmatch)

In [ ]:
p1 = np.float32([ kp1[m.queryIdx].pt for m in good ]).reshape(-1,1,2)
p2 = np.float32([ kp2[m.trainIdx].pt for m in good ]).reshape(-1,1,2)

print(f"Extracted {len(p1)} points from img1 (p1) and {len(p2)} points from img2 (p2).")
print("Camera matrix and distortion coefficients are ready for use.")

In [ ]:
p1_undistorted = cv2.undistortPoints(p1, camera_matrix, dist_coeffs, P=camera_matrix)
p2_undistorted = cv2.undistortPoints(p2, camera_matrix, dist_coeffs, P=camera_matrix)

# Reshape to 2D array for cv2.findEssentialMat
p1_undistorted_flat = p1_undistorted.reshape(-1, 2)
p2_undistorted_flat = p2_undistorted.reshape(-1, 2)

# Estimate Essential Matrix
E, mask_E = cv2.findEssentialMat(p1_undistorted_flat, p2_undistorted_flat, camera_matrix, method=cv2.RANSAC, prob=0.999, threshold=1.0)

# Recover Pose
_, R_rel, t_rel, mask_pose = cv2.recoverPose(E, p1_undistorted_flat, p2_undistorted_flat, camera_matrix, mask=mask_E)

print("Matched points undistorted.")
print("Essential Matrix estimated.")
print("Relative Rotation (R_rel) and Translation (t_rel) recovered.")
print("R_rel:\n", R_rel)
print("t_rel:\n", t_rel)

> **⚠️ Scale ambiguity (important!)**
>
> `cv2.recoverPose` returns a translation `t_rel` that is a **unit vector**
> (`‖t_rel‖ = 1`). A single camera cannot tell the difference between a small
> object up close and a large object far away, so monocular VO recovers the
> **direction** of motion but not its magnitude.
>
> Consequence: the trajectory shape is correct, but it is **not metric** — you
> can't read distances in meters off the plot. Real systems fix this with a
> stereo rig, wheel odometry, IMU, or a known object size. We'll just assume a
> unit step per frame here.

In [ ]:
import matplotlib.pyplot as plt

# Define the initial camera position
initial_camera_pos = (0, 0)

# Extract X and Z components from t_rel
# t_rel is a 3x1 vector: [[X], [Y], [Z]]
# For a top-down view, we are interested in X (lateral) and Z (forward) movement.
# The coordinate system assumes Z is forward, X is right, and Y is down.
# So, t_rel[0] is X, t_rel[2] is Z.
second_camera_x = t_rel[0, 0]
second_camera_z = t_rel[2, 0]

# Create a new Matplotlib figure
plt.figure(figsize=(8, 8))

# Plot the initial camera position
plt.plot(initial_camera_pos[0], initial_camera_pos[1], 'ro', markersize=10, label='Camera 1 Position (Origin)')

# Plot the second camera's position
plt.plot(second_camera_x, second_camera_z, 'go', markersize=10, label='Camera 2 Position')

# Draw a dashed blue line connecting the two positions
plt.plot([initial_camera_pos[0], second_camera_x], [initial_camera_pos[1], second_camera_z], 'b--', label='Relative Translation')

# Set labels and title
plt.xlabel('X-coordinate (Lateral Movement)')
plt.ylabel('Z-coordinate (Forward Movement)')
plt.title('2D Plot of Camera Positions for Two Frames (Top-Down View)')

# Add a grid
plt.grid(True)

# Ensure equal scaling for the X and Z axes
plt.axis('equal')

# Display the legend
plt.legend()

# Show the plot
plt.show()

In [ ]:
trajectory_points = []
trajectory_points.append(t_rel.flatten())

### **Exercise: Add a 3rd Camera?**

In [ ]:
img3 = cv2.imread("downtown/front_images_downtown/1557197712048522.jpg")
# DETECTOR
kp3 = #TODO
# DESCRIPTOR
kp3, des3 = #TODO
img3_kp = #TODO

plt.imshow(to_rgb(img3_kp))
plt.show()

In [ ]:
### MATCH
matches = #TODO

good = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        #TODO

print(f"Found {len(good)} good matches after Lowe's ratio test.")
img_briefmatch23 = #TODO
cv2_imshow(img_briefmatch23)

In [ ]:
# The 'good' matches here refer to the matches between des2 and des3 from the previous cell OilWm56R-YYj

# Extract points from kp2 using queryIdx from the good matches (des2 -> des3)
p2_for_img2img3 = #TODO
p3_for_img2img3 = #TODO

# Undistort points for the current match
p2_undistorted_for_img2img3 = #TODO
p3_undistorted_for_img2img3 = #TODO

# Reshape to 2D array for cv2.findEssentialMat
p2_undistorted_flat_for_img2img3 = #TODO
p3_undistorted_flat_for_img2img3 = #TODO

# Estimate Essential Matrix
E, mask_E = #TODO
# Recover Pose
_, R_rel_23, t_rel_23, mask_pose = #TODO

print("Recovered Relative Rotation Matrix (R_rel_23):\n", R_rel_23)
print("Recovered Relative Translation Vector (t_rel_23):\n", t_rel_23)


In [ ]:
# EXERCISE: accumulate the trajectory CORRECTLY.
#
# A relative pose (R_rel, t_rel) maps points from the previous camera frame
# into the current one. To place each camera in a common WORLD frame you must
# CHAIN the transforms, not just add the translation vectors:
#
#     t_world = t_world + R_world @ t_rel
#     R_world = R_rel @ R_world
#
# (Simply summing t_rel ignores how the camera rotated, so the path breaks
#  the moment the car turns.)
#
# Reminder: monocular VO recovers t_rel only up to scale (||t_rel|| == 1),
# so distances are NOT metric here — only the SHAPE of the path is meaningful.

poses = []

R_world = np.eye(3)
t_world = np.zeros((3, 1))
poses.append(t_world.flatten())

# Camera 1 -> Camera 2
t_world = #TODO
R_world = #TODO
poses.append(t_world.flatten())

# Camera 2 -> Camera 3
t_world = #TODO
R_world = #TODO
poses.append(t_world.flatten())

poses = np.array(poses)

plt.figure(figsize=(8, 8))
plt.plot(poses[:, 0], poses[:, 2], 'k--', label='Trajectory Path')
plt.scatter(poses[:, 0], poses[:, 2], c=['r', 'g', 'b'], s=90, zorder=3)
for i, (x, _, z) in enumerate(poses):
    plt.annotate(f'Cam {i + 1}', (x, z), textcoords="offset points", xytext=(8, 8))
plt.xlabel('X (lateral movement)')
plt.ylabel('Z (forward movement)')
plt.title('Accumulated Camera Positions (Top-Down) — Proper Pose Chaining')
plt.grid(True)
plt.axis('equal')
plt.legend()
plt.show()

---

### **Solution**

In [ ]:
img3 = cv2.imread("downtown/front_images_downtown/1557197712048522.jpg")
# DETECTOR
kp3, des3 = orb.detectAndCompute(img3,None)
img3_kp = cv2.drawKeypoints(img3, kp3, None, color=(0,255,0), flags=0)

plt.imshow(to_rgb(img3_kp))
plt.show()

In [ ]:
### MATCH
matches = flann.knnMatch(np.float32(des2), np.float32(des3), k=2)

good = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        good.append(m)

print(f"Found {len(good)} good matches after Lowe's ratio test.")
img_briefmatch23 = cv2.drawMatches(img2,kp2,img3,kp3,good,None,**draw_params)
cv2_imshow(img_briefmatch23)

In [ ]:
# The 'good' matches here refer to the matches between des2 and des3 from the previous cell OilWm56R-YYj

# Extract points from kp2 using queryIdx from the good matches (des2 -> des3)
p2_for_img2img3 = np.float32([ kp2[m.queryIdx].pt for m in good ]).reshape(-1,1,2)
p3_for_img2img3 = np.float32([ kp3[m.trainIdx].pt for m in good ]).reshape(-1,1,2)

# Undistort points for the current match
p2_undistorted_for_img2img3 = cv2.undistortPoints(p2_for_img2img3, camera_matrix, dist_coeffs, P=camera_matrix)
p3_undistorted_for_img2img3 = cv2.undistortPoints(p3_for_img2img3, camera_matrix, dist_coeffs, P=camera_matrix)

# Reshape to 2D array for cv2.findEssentialMat
p2_undistorted_flat_for_img2img3 = p2_undistorted_for_img2img3.reshape(-1, 2)
p3_undistorted_flat_for_img2img3 = p3_undistorted_for_img2img3.reshape(-1, 2)

# Estimate Essential Matrix
E, mask_E = cv2.findEssentialMat(p2_undistorted_flat_for_img2img3, p3_undistorted_flat_for_img2img3, camera_matrix, method=cv2.RANSAC, prob=0.999, threshold=1.0)

# Recover Pose
_, R_rel_23, t_rel_23, mask_pose = cv2.recoverPose(E, p2_undistorted_flat_for_img2img3, p3_undistorted_flat_for_img2img3, camera_matrix, mask=mask_E)

print("Recovered Relative Rotation Matrix (R_rel_23):\n", R_rel_23)
print("Recovered Relative Translation Vector (t_rel_23):\n", t_rel_23)


In [ ]:
# SOLUTION: correct pose accumulation via transform chaining.
poses = []

R_world = np.eye(3)
t_world = np.zeros((3, 1))
poses.append(t_world.flatten())

# Camera 1 -> Camera 2
t_world = t_world + R_world @ t_rel
R_world = R_rel @ R_world
poses.append(t_world.flatten())

# Camera 2 -> Camera 3
t_world = t_world + R_world @ t_rel_23
R_world = R_rel_23 @ R_world
poses.append(t_world.flatten())

poses = np.array(poses)

plt.figure(figsize=(8, 8))
plt.plot(poses[:, 0], poses[:, 2], 'k--', label='Trajectory Path')
plt.scatter(poses[:, 0], poses[:, 2], c=['r', 'g', 'b'], s=90, zorder=3)
for i, (x, _, z) in enumerate(poses):
    plt.annotate(f'Cam {i + 1}', (x, z), textcoords="offset points", xytext=(8, 8))
plt.xlabel('X (lateral movement)')
plt.ylabel('Z (forward movement)')
plt.title('Accumulated Camera Positions (Top-Down) — Proper Pose Chaining')
plt.grid(True)
plt.axis('equal')
plt.legend()
plt.show()

## **Video Challenge: Full Visual Odometry on a Real Sequence**

Two frames are a warm-up. Real Visual Odometry runs the pipeline over an
**entire video**, chaining every relative pose into one continuous trajectory.

For each consecutive frame pair `(prev → curr)` you will:

1. **Detect + describe** ORB features on the new frame
2. **Match** against the previous frame (FLANN + Lowe's ratio test)
3. **Undistort** the matched points using the camera calibration
4. **Estimate** the Essential matrix (RANSAC) and **recover** `R_rel, t_rel`
5. **Accumulate** the global pose by *chaining* (same math as the 3-camera exercise)

The cell below sets up everything you need. Then complete the exercise loop —
the worked solution and a polished trajectory plot follow.

In [ ]:
import cv2
import numpy as np
import json
import matplotlib.pyplot as plt

# Global camera pose, expressed in the world frame
R_world = np.eye(3)            # 3x3 rotation, starts at identity
t_world = np.zeros((3, 1))     # 3x1 translation, starts at the origin

# Trajectory storage (list of camera positions in the world frame)
trajectory_points = [t_world.flatten()]

# Video + calibration for the "city" sequence
video_path = 'city/front_camera_city.mp4'
calibration_json_path = 'city/camera_calibration_city.json'

with open(calibration_json_path, 'r') as f:
    calibration_data = json.load(f)
camera_matrix = np.array(calibration_data['intrinsics'])
dist_coeffs = np.array(calibration_data['distortion'])

# ORB detector/descriptor (same as the two-frame pipeline)
orb = cv2.ORB_create(nfeatures=2000)

# FLANN matcher
FLANN_INDEX_KDTREE = 0
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)
flann = cv2.FlannBasedMatcher(index_params, search_params)

print("Visual Odometry pipeline initialized:")
print(f"  video       : {video_path}")
print(f"  calibration : {calibration_json_path}")
print(f"  detector    : ORB (2000 features) + FLANN matcher")

In [ ]:
# EXERCISE: run Visual Odometry over the whole video.
# Fill in every #TODO. Re-run the setup cell above first so the pose resets.

cap = cv2.VideoCapture(video_path)

ret, prev_frame = cap.read()
assert ret, "Could not read the first frame of the video."

# Features on the very first frame
kp_prev, des_prev = #TODO  detect + describe on prev_frame

frame_idx = 0
while True:
    ret, curr_frame = cap.read()
    if not ret:
        break
    frame_idx += 1

    kp_curr, des_curr = #TODO  detect + describe on curr_frame

    # Skip degenerate frames (nothing to match against)
    if des_prev is None or des_curr is None or len(kp_curr) < 8:
        prev_frame, kp_prev, des_prev = curr_frame, kp_curr, des_curr
        continue

    matches = #TODO  flann.knnMatch(...) with k=2
    good = []
    for pair in matches:
        if len(pair) != 2:          # FLANN can return <2 neighbours
            continue
        m, n = pair
        if m.distance < 0.7 * n.distance:
            good.append(m)

    if len(good) < 8:               # need >=5 for the Essential matrix; keep margin
        prev_frame, kp_prev, des_prev = curr_frame, kp_curr, des_curr
        continue

    p_prev = #TODO  points from kp_prev via m.queryIdx, shape (-1, 1, 2)
    p_curr = #TODO  points from kp_curr via m.trainIdx, shape (-1, 1, 2)

    p_prev_u = #TODO  cv2.undistortPoints(...) then reshape(-1, 2)
    p_curr_u = #TODO

    E, mask_E = #TODO  cv2.findEssentialMat(... RANSAC ...)
    if E is None or E.shape != (3, 3):
        prev_frame, kp_prev, des_prev = curr_frame, kp_curr, des_curr
        continue

    _, R_rel, t_rel, _ = #TODO  cv2.recoverPose(...)

    # Accumulate the global pose (monocular -> assume unit scale)
    #TODO  update t_world and R_world by chaining (see the 3-camera solution)
    trajectory_points.append(t_world.flatten())

    prev_frame, kp_prev, des_prev = curr_frame, kp_curr, des_curr

cap.release()
trajectory_points = np.array(trajectory_points)
print(f"Processed {frame_idx} frames -> {len(trajectory_points)} poses estimated.")

---
### **Solution**

In [ ]:
# SOLUTION: full Visual Odometry over the video.
# Re-run the setup cell first so R_world / t_world / trajectory_points reset.

cap = cv2.VideoCapture(video_path)

ret, prev_frame = cap.read()
assert ret, "Could not read the first frame of the video."

kp_prev, des_prev = orb.detectAndCompute(prev_frame, None)

frame_idx = 0
used = 0
while True:
    ret, curr_frame = cap.read()
    if not ret:
        break
    frame_idx += 1

    kp_curr, des_curr = orb.detectAndCompute(curr_frame, None)

    if des_prev is None or des_curr is None or len(kp_curr) < 8:
        prev_frame, kp_prev, des_prev = curr_frame, kp_curr, des_curr
        continue

    matches = flann.knnMatch(np.float32(des_prev), np.float32(des_curr), k=2)
    good = []
    for pair in matches:
        if len(pair) != 2:
            continue
        m, n = pair
        if m.distance < 0.7 * n.distance:
            good.append(m)

    if len(good) < 8:
        prev_frame, kp_prev, des_prev = curr_frame, kp_curr, des_curr
        continue

    p_prev = np.float32([kp_prev[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    p_curr = np.float32([kp_curr[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

    p_prev_u = cv2.undistortPoints(p_prev, camera_matrix, dist_coeffs, P=camera_matrix).reshape(-1, 2)
    p_curr_u = cv2.undistortPoints(p_curr, camera_matrix, dist_coeffs, P=camera_matrix).reshape(-1, 2)

    E, mask_E = cv2.findEssentialMat(p_prev_u, p_curr_u, camera_matrix,
                                     method=cv2.RANSAC, prob=0.999, threshold=1.0)
    if E is None or E.shape != (3, 3):
        prev_frame, kp_prev, des_prev = curr_frame, kp_curr, des_curr
        continue

    _, R_rel, t_rel, _ = cv2.recoverPose(E, p_prev_u, p_curr_u, camera_matrix, mask=mask_E)

    # Chain the relative pose into the global frame (monocular -> unit scale)
    t_world = t_world + R_world @ t_rel
    R_world = R_rel @ R_world
    trajectory_points.append(t_world.flatten())
    used += 1

    prev_frame, kp_prev, des_prev = curr_frame, kp_curr, des_curr

cap.release()
trajectory_points = np.array(trajectory_points)
print(f"Processed {frame_idx} frames, used {used} pairs -> {len(trajectory_points)} trajectory points.")

### **Live Side-by-Side: Camera Feed + Odometry**

Instead of a single static plot, let's render a video that mimics what a real
SLAM system shows on screen:

- **Left panel** — the camera feed with the tracked features drawn on it
  (green lines show how each point moved between frames).
- **Right panel** — the odometry map building up in real time as the car drives.

We do this in **two passes**: first run the VO to learn the full trajectory (so
the map has stable, non-jittering axes), then composite the two panels
frame-by-frame into an MP4 and play it inline.

In [ ]:
import cv2
import numpy as np

# ------------------------------------------------------------------ #
# PASS 1 — run VO over the whole clip, remembering, for every video    #
# frame: the camera position so far, and the matched feature points    #
# (so we can draw the tracks later without re-running ORB).            #
# ------------------------------------------------------------------ #
R_world = np.eye(3)
t_world = np.zeros((3, 1))

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 10.0

ret, prev_frame = cap.read()
assert ret, "Could not read the first frame of the video."
H, W = prev_frame.shape[:2]
kp_prev, des_prev = orb.detectAndCompute(prev_frame, None)

positions = [(float(t_world[0, 0]), float(t_world[2, 0]))]  # (x, z) per frame
tracks = [None]                                             # (pts_prev, pts_curr) per frame

while True:
    ret, curr_frame = cap.read()
    if not ret:
        break

    kp_curr, des_curr = orb.detectAndCompute(curr_frame, None)
    track = None

    if des_prev is not None and des_curr is not None and len(kp_curr) >= 8:
        matches = flann.knnMatch(np.float32(des_prev), np.float32(des_curr), k=2)
        good = []
        for pair in matches:
            if len(pair) != 2:
                continue
            m, n = pair
            if m.distance < 0.7 * n.distance:
                good.append(m)

        if len(good) >= 8:
            pts_prev = np.float32([kp_prev[m.queryIdx].pt for m in good])
            pts_curr = np.float32([kp_curr[m.trainIdx].pt for m in good])

            p_prev_u = cv2.undistortPoints(pts_prev.reshape(-1, 1, 2),
                                           camera_matrix, dist_coeffs, P=camera_matrix).reshape(-1, 2)
            p_curr_u = cv2.undistortPoints(pts_curr.reshape(-1, 1, 2),
                                           camera_matrix, dist_coeffs, P=camera_matrix).reshape(-1, 2)

            E, mask_E = cv2.findEssentialMat(p_prev_u, p_curr_u, camera_matrix,
                                             method=cv2.RANSAC, prob=0.999, threshold=1.0)
            if E is not None and E.shape == (3, 3):
                _, R_rel, t_rel, _ = cv2.recoverPose(E, p_prev_u, p_curr_u, camera_matrix, mask=mask_E)
                t_world = t_world + R_world @ t_rel
                R_world = R_rel @ R_world
                track = (pts_prev, pts_curr)

    positions.append((float(t_world[0, 0]), float(t_world[2, 0])))
    tracks.append(track)
    prev_frame, kp_prev, des_prev = curr_frame, kp_curr, des_curr

cap.release()
positions = np.array(positions)
print(f"Pass 1 done: {len(positions)} frames, trajectory computed.")

# ------------------------------------------------------------------ #
# Fixed world -> pixel mapping for the odometry panel (no jitter).     #
# ------------------------------------------------------------------ #
xs, zs = positions[:, 0], positions[:, 1]
xmin, xmax = xs.min(), xs.max()
zmin, zmax = zs.min(), zs.max()
dx = (xmax - xmin) or 1.0
dz = (zmax - zmin) or 1.0
xmin, xmax = xmin - 0.1 * dx, xmax + 0.1 * dx
zmin, zmax = zmin - 0.1 * dz, zmax + 0.1 * dz
M = 40  # pixel margin

def world_to_px(x, z):
    u = int((x - xmin) / (xmax - xmin) * (W - 2 * M) + M)
    v = int((1.0 - (z - zmin) / (zmax - zmin)) * (H - 2 * M) + M)  # z up
    return u, v

px_path = [world_to_px(x, z) for x, z in zip(xs, zs)]

# ------------------------------------------------------------------ #
# Dark-mode helpers.                                                  #
#   Colours are BGR. Theme: deep navy bg + neon cyan/green accents.    #
# ------------------------------------------------------------------ #
BG        = (28, 22, 18)     # near-black blue
GRID      = (52, 42, 34)     # faint grid
CYAN      = (255, 209, 64)   # neon path
CYAN_DIM  = (120, 96, 28)    # glow underlay
GREEN     = (120, 255, 80)   # start / features
RED       = (90, 90, 255)    # current position
TEXT      = (235, 235, 235)

def hud_text(img, text, org, scale=0.7, color=TEXT):
    """Text on a translucent dark pill for legibility on any background."""
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
    x, y = org
    pad = 8
    overlay = img.copy()
    cv2.rectangle(overlay, (x - pad, y - th - pad), (x + tw + pad, y + pad), (15, 12, 10), -1)
    cv2.addWeighted(overlay, 0.55, img, 0.45, 0, img)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, 2, cv2.LINE_AA)

def glow_polyline(img, pts):
    """Neon path: a dim thick underlay + a bright thin core."""
    arr = np.array(pts, np.int32)
    cv2.polylines(img, [arr], False, CYAN_DIM, 7, cv2.LINE_AA)
    cv2.polylines(img, [arr], False, CYAN, 2, cv2.LINE_AA)

def glow_dot(img, center, color, r=6):
    cv2.circle(img, center, r + 4, tuple(int(c * 0.45) for c in color), -1, cv2.LINE_AA)
    cv2.circle(img, center, r, color, -1, cv2.LINE_AA)

# Pre-build the dark odometry backdrop with a subtle grid (drawn once).
odo_bg = np.full((H, W, 3), BG, np.uint8)
for gx in range(M, W - M + 1, max(1, (W - 2 * M) // 8)):
    cv2.line(odo_bg, (gx, M), (gx, H - M), GRID, 1, cv2.LINE_AA)
for gy in range(M, H - M + 1, max(1, (H - 2 * M) // 8)):
    cv2.line(odo_bg, (M, gy), (W - M, gy), GRID, 1, cv2.LINE_AA)
cv2.rectangle(odo_bg, (M, M), (W - M, H - M), GRID, 1, cv2.LINE_AA)

# ------------------------------------------------------------------ #
# PASS 2 — re-read the video (decode only, no ORB) and composite.      #
# ------------------------------------------------------------------ #
out_path = 'output/vo_sidebyside.mp4'
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W * 2, H))

cap = cv2.VideoCapture(video_path)
n_frames = len(tracks)
idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # --- Left: camera, darkened, with neon feature tracks ---
    cam = (frame * 0.78).astype(np.uint8)  # dim the feed for the dark theme
    tr = tracks[idx] if idx < len(tracks) else None
    if tr is not None:
        pts_prev, pts_curr = tr
        for (x0, y0), (x1, y1) in zip(pts_prev[:200], pts_curr[:200]):
            cv2.line(cam, (int(x0), int(y0)), (int(x1), int(y1)), GREEN, 1, cv2.LINE_AA)
            cv2.circle(cam, (int(x1), int(y1)), 2, GREEN, -1, cv2.LINE_AA)
    n_feat = 0 if tr is None else len(tr[0])
    hud_text(cam, f"CAMERA + FEATURES", (15, 34), 0.7)
    hud_text(cam, f"frame {idx:04d}  |  {n_feat} tracks", (15, H - 18), 0.55)

    # --- Right: dark odometry map building up ---
    odo = odo_bg.copy()
    if idx >= 1:
        glow_polyline(odo, px_path[:idx + 1])
    glow_dot(odo, px_path[0], GREEN, 6)
    glow_dot(odo, px_path[idx], RED, 7)
    hud_text(odo, "VISUAL ODOMETRY", (15, 34), 0.7, CYAN)
    hud_text(odo, "top-down  |  unit-scale (not metric)", (15, H - 18), 0.55)

    # progress bar along the bottom
    bx0, bx1, by = M, W - M, H - 6
    cv2.line(odo, (bx0, by), (bx1, by), GRID, 3, cv2.LINE_AA)
    prog = int(bx0 + (bx1 - bx0) * (idx / max(1, n_frames - 1)))
    cv2.line(odo, (bx0, by), (prog, by), CYAN, 3, cv2.LINE_AA)

    canvas = np.hstack([cam, odo])
    cv2.line(canvas, (W, 0), (W, H), (10, 10, 10), 2)  # divider
    writer.write(canvas)
    idx += 1

cap.release()
writer.release()
print(f"Pass 2 done: wrote {idx} frames -> {out_path}")

In [ ]:
# Play the side-by-side result inline (re-encode to H.264 so the browser
# can play it; fall back to the raw mp4v file if ffmpeg isn't available).
import os
from base64 import b64encode
from IPython.display import HTML, display

src = 'output/vo_sidebyside.mp4'
play = 'output/vo_sidebyside_h264.mp4'
if os.system(f'ffmpeg -y -loglevel error -i {src} -vcodec libx264 -pix_fmt yuv420p {play}') != 0:
    play = src

b64 = b64encode(open(play, 'rb').read()).decode()
display(HTML(f'''
<div style="background:#0c0a09;padding:16px;border-radius:12px;
            box-shadow:0 0 24px #000;display:inline-block">
  <video width="900" autoplay loop controls
         style="border-radius:8px;display:block">
    <source src="data:video/mp4;base64,{b64}" type="video/mp4">
  </video>
</div>'''))